# Research 56 — Fresh-fold platform/device interaction synthesis

Compare one predeclared fixed 80/10/10 blend (reference + loyalty×platform + engagement×device) on the existing seed-42 predictions and fresh seed-2025 predictions. Record full OOF and fold-slice macro/weighted F1. No weight search or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), filename
    return probs / row_sums


p_base = load_probs("segment-oof.csv")
p_loyalty_42 = load_probs("segment-lightgbm-loyalty-platform-oof.csv")
p_device_42 = load_probs("segment-lightgbm-engagement-device-oof.csv")
p_loyalty_2025 = load_probs("segment-lightgbm-loyalty-platform-seed2025-oof.csv")
p_device_2025 = load_probs("segment-lightgbm-engagement-device-seed2025-oof.csv")
p_mix_42 = 0.8 * p_base + 0.1 * p_loyalty_42 + 0.1 * p_device_42
p_mix_2025 = 0.8 * p_base + 0.1 * p_loyalty_2025 + 0.1 * p_device_2025

rows = []
for seed, probs in [(42, p_mix_42), (2025, p_mix_2025)]:
    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    pred = probs.argmax(axis=1)
    row = {
        "candidate": "Fixed 80/10/10 reference + loyalty×platform + engagement×device",
        "cv_seed": seed,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        score = f1_score(y[valid_idx], pred[valid_idx], average="macro")
        baseline = f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro")
        row[f"fold_{fold}_macro_f1"] = score
        row[f"fold_{fold}_delta_vs_reference"] = score - baseline
    rows.append(row)

results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-platform-device-fresh-fold-screen.csv", index=False)
for probs, filename in [
    (p_mix_42, "segment-platform-device-seed42-threeway-oof.csv"),
    (p_mix_2025, "segment-platform-device-seed2025-threeway-oof.csv"),
]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / "predictions" / filename, index=False
    )
print("Fixed 80/10/10 mixtures evaluated on two CV partitions; no tuning; Test not loaded.")

,candidate,cv_seed,macro_f1,weighted_f1,fold_1_macro_f1,fold_1_delta_vs_reference,fold_2_macro_f1,fold_2_delta_vs_reference,fold_3_macro_f1,fold_3_delta_vs_reference
0,Fixed 80/10/10 reference + loyalty×platform + ...,42,0.751460,0.764942,0.750046,0.000601,0.750416,0.000595,0.753891,0.000501
1,Fixed 80/10/10 reference + loyalty×platform + ...,2025,0.751955,0.765529,0.751638,0.000841,0.750923,0.001512,0.753291,0.000829


Fixed 80/10/10 mixtures evaluated on two CV partitions; no tuning; Test not loaded.
